In [13]:

import pandas as pd
import joblib

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [14]:
fighters_df = pd.read_csv('/home/duyle/Documents/VSC/Project_DAP391/processed_data/fighters_w_image_2.csv')
stats_df = pd.read_csv('/home/duyle/Documents/VSC/Project_DAP391/processed_data/fight_stats_with_weghtclass_date_location.csv')
results_df = pd.read_csv('/home/duyle/Documents/VSC/Project_DAP391/processed_data/fight_results_with_locale_2.csv')


In [15]:
fighters_df['Name'] = fighters_df['Name'].str.strip().str.lower()
stats_df['ROUND'] = stats_df['ROUND'].str.replace('Round ','')
results_df['DATE'] = pd.to_datetime(results_df['DATE'])

In [16]:
fighters_df.columns

Index(['Name', 'Nickname', 'Weight_Class', 'Knockouts', 'Submissions',
       'First_Round_Finishes', 'Striking_Accuracy', 'Takedown_Accuracy',
       'Sig_Str_Attempted Total', 'Takedowns_Landed_Total',
       'Takedowns_Attempted_Total', 'Sig_Strikes_Per Min',
       'Takedown_Avg_Per Min', 'Sig_Str_Def', 'Knockdown_Avg',
       'Sig_Strikes_Absorbed_Per_Min', 'Sub_Avg_Per_Min', 'Takedown_Def',
       'Avg_Fight_Time', 'Place_of_Birth', 'Octagon_Debut', 'Wins', 'Losses',
       'Draws', 'Sig_Str_Landed Total_Total', 'Sig_Str_Landed Total_Percent',
       'Sig_Strikes_While_Clinched_Total',
       'Sig_Strikes_While_Clinched_Percent',
       'Sig_Strikes_While_Standing_Total',
       'Sig_Strikes_While_Standing_Percent',
       'Sig_Strikes_While_Grounded_Total',
       'Sig_Strikes_While_Grounded_Percent', 'Sig_Strikes_Head_Total',
       'Sig_Strikes_Head_Percent', 'Sig_Strikes_Body_Total',
       'Sig_Strikes_Body_Percent', 'Sig_Strikes_Leg_Total',
       'Sig_Strikes_Leg_Percent',

In [17]:
stats_df.columns

Index(['EVENT', 'BOUT', 'ROUND', 'FIGHTER', 'KD', 'SIG.STR. %', 'TD %',
       'SUB.ATT', 'REV.', 'CTRL', 'sig_str_attempt', 'sig_str_land',
       'total_str_attempt', 'total_str_land', 'touchdown_attempt',
       'takedown_land', 'total_leg_attempt', 'total_leg_land',
       'total_distance_strike_attempt', 'total_distance_strike_land',
       'total_body_attempt', 'total_body_land', 'total_clinch_attempt',
       'total_clinch_land', 'total_ground_attempt', 'total_ground_land',
       'total_head_attempt', 'total_head_land', 'weightclass_numeric',
       'weight_class', 'gender', 'is_title_bout', 'DATE', 'LOCATION'],
      dtype='object')

In [18]:
def create_matchup_features(fighter1, fighter2):
    """Create symmetric matchup features that don't depend on fighter order."""
    features = {}
    numerical_features = [
        'Striking_Accuracy', 'Takedown_Accuracy', 'Sig_Str_Def', 'Takedown_Def',
        'Takedown_Avg_Per Min', 'Knockdown_Avg'
    ]
    
    for feature in numerical_features:
        if feature in fighter1 and feature in fighter2:
            # Absolute difference (symmetric)
            features[f'{feature}_abs_diff'] = abs(fighter1[feature] - fighter2[feature])
            
            
    # Calculate win rates
    f1_total = fighter1['Wins'] + fighter1['Losses']
    f2_total = fighter2['Wins'] + fighter2['Losses']
    
    f1_win_rate = 0 if f1_total == 0 else fighter1['Wins'] / f1_total
    f2_win_rate = 0 if f2_total == 0 else fighter2['Wins'] / f2_total
    
    # Symmetric win rate features
    features['win_rate_abs_diff'] = abs(f1_win_rate - f2_win_rate)
    

    
    # Experience gap (always symmetric)
    debut1 = pd.to_datetime(fighter1['Octagon_Debut'])
    debut2 = pd.to_datetime(fighter2['Octagon_Debut'])
    features['days_debut'] = abs((debut1 - debut2).days)
    
    return features

In [19]:
def create_recency_weighted_features(fighter_name, results_df, decay_factor=0.9):
    
    fighter_history = results_df[(results_df['FIGHTER_1'] == fighter_name) | 
                                (results_df['FIGHTER_2'] == fighter_name)]
    
    if fighter_history.empty:
        return {
            'recent_win_rate': 0,
            'decay_weighted_win_rate': 0,
            'form_momentum': 0
        }
    
    # Sort by date (most recent first)
    fighter_history = fighter_history.sort_values(by='DATE', ascending=False)
    
    # Get wins/losses and calculate weighted stats
    wins = []
    for idx, fight in fighter_history.iterrows():
        if (fight['FIGHTER_1'] == fighter_name and fight['fighter_1_result'] == 1) or \
           (fight['FIGHTER_2'] == fighter_name and fight['fighter_2_result'] == 1):
            wins.append(1)
        else:
            wins.append(0)
    
    # Calculate recency-weighted win rate
    weights = [decay_factor**i for i in range(len(wins))]
    weighted_wins = sum(w*win for w, win in zip(weights, wins))
    weighted_total = sum(weights)
    decay_weighted_win_rate = weighted_wins / weighted_total if weighted_total > 0 else 0
    
    # Recent win rate (last 3 fights)
    recent_win_rate = sum(wins[:3]) / min(3, len(wins))
    
    # Form momentum (difference between recent and overall win rates)
    all_time_win_rate = sum(wins) / len(wins)
    form_momentum = recent_win_rate - all_time_win_rate
    
    return {
        'recent_win_rate': recent_win_rate,
        'decay_weighted_win_rate': decay_weighted_win_rate,
        'form_momentum': form_momentum
    }


In [20]:
def create_career_features(fighter1, fighter2):
    """Create symmetric career features that don't depend on fighter order."""
    features = {}
    
    fighter1_info = fighters_df[fighters_df['Name'] == fighter1]
    fighter2_info = fighters_df[fighters_df['Name'] == fighter2]
    
    if fighter1_info.empty or fighter2_info.empty:
        if fighter1_info.empty:
            print(f"Fighter {fighter1} not found")
        if fighter2_info.empty:
            print(f"Fighter {fighter2} not found")
        return features
    
    f1_career_wins = fighter1_info['Wins'].iloc[0]
    f1_career_losses = fighter1_info['Losses'].iloc[0]
    f1_career_draws = fighter1_info['Draws'].iloc[0]
    
    f2_career_wins = fighter2_info['Wins'].iloc[0]
    f2_career_losses = fighter2_info['Losses'].iloc[0]
    f2_career_draws = fighter2_info['Draws'].iloc[0]
    
    f1_total_career_fights = f1_career_wins + f1_career_losses
    f1_career_win_rate = f1_career_wins / f1_total_career_fights if f1_total_career_fights > 0 else 0
    
    f2_total_career_fights = f2_career_wins + f2_career_losses
    f2_career_win_rate = f2_career_wins / f2_total_career_fights if f2_total_career_fights > 0 else 0
    
    # Experience differences (symmetric)
    f1_experience = f1_career_wins + f1_career_losses + f1_career_draws
    f2_experience = f2_career_wins + f2_career_losses + f2_career_draws
    
    features['experience_abs_diff'] = abs(f1_experience - f2_experience)

    
    # Win rate comparison (symmetric)
    features['win_rate_abs_diff'] = abs(f1_career_win_rate - f2_career_win_rate)
    

    
    # Wins and losses comparison (symmetric)
    features['wins_abs_diff'] = abs(f1_career_wins - f2_career_wins)

    
    features['losses_abs_diff'] = abs(f1_career_losses - f2_career_losses)
    

    # Momentum calculation (symmetric)
    f1_history = results_df[(results_df['FIGHTER_1'] == fighter1) | (results_df['FIGHTER_2'] == fighter1)]
    f2_history = results_df[(results_df['FIGHTER_1'] == fighter2) | (results_df['FIGHTER_2'] == fighter2)]
    
    f1_momentum = 0
    f2_momentum = 0
    
    # Process fighter 1 recent history
    if len(f1_history) >= 2:
        f1_history = f1_history.sort_values(by='DATE', ascending=False)
        recent_f1_fights = f1_history.head(2)
        
        f1_recent_wins = 0
        for _, fight in recent_f1_fights.iterrows():
            if (fight['FIGHTER_1'] == fighter1 and fight['fighter_1_result'] == 1) or \
               (fight['FIGHTER_2'] == fighter1 and fight['fighter_1_result'] == 0):
                f1_recent_wins += 1
        
        f1_recent_win_rate = f1_recent_wins / 2
        f1_momentum = f1_recent_win_rate - f1_career_win_rate
    
    # Process fighter 2 recent history
    if len(f2_history) >= 2:
        f2_history = f2_history.sort_values(by='DATE', ascending=False)
        recent_f2_fights = f2_history.head(2)
        
        f2_recent_wins = 0
        for _, fight in recent_f2_fights.iterrows():
            if (fight['FIGHTER_1'] == fighter2 and fight['fighter_1_result'] == 1) or \
               (fight['FIGHTER_2'] == fighter2 and fight['fighter_1_result'] == 0):
                f2_recent_wins += 1
        
        f2_recent_win_rate = f2_recent_wins / 2
        f2_momentum = f2_recent_win_rate - f2_career_win_rate
    
    # Momentum comparison (symmetric)
    features['momentum_abs_diff'] = abs(f1_momentum - f2_momentum)

    
    return features

In [21]:
def create_weight_class_relative_features(fighter, weight_class_stats_df):
    """Create features comparing fighter stats to weight class averages."""
    
    weight_class = fighter['Weight_Class']
    
    # Get weight class averages
    wc_avg = weight_class_stats_df[weight_class_stats_df['weight_class'] == weight_class].iloc[0]
    
    # Calculate relative features (how fighter compares to weight class average)
    rel_features = {}
    numerical_stats = ['Striking_Accuracy', 'Takedown_Accuracy', 'Sig_Strikes_Per Min', 
                      'Takedown_Avg_Per Min', 'Sig_Str_Def', 'Knockdown_Avg']
    
    for stat in numerical_stats:
        if stat in fighter and stat in wc_avg:
            # Convert to percentage above/below average
            rel_features[f'{stat}_rel_to_class'] = (fighter[stat] / wc_avg[stat]) - 1 
    
    return rel_features

# Create weight class statistics dataframe (this should be done once)
def calculate_weight_class_averages(fighters_df):
    """Calculate average statistics by weight class for normalization."""
    # Group by weight class and calculate means
    return fighters_df.groupby('Weight_Class').mean()


In [22]:
def create_style_matchup_features(fighter1, fighter2):
    """Create symmetric style features that don't depend on fighter order."""
    features = {}
    
    # Calculate strike bias
    fighter1_strike_bias = fighter1['Sig_Strikes_Per Min'] / (fighter1['Takedown_Avg_Per Min'] + 0.1)
    fighter2_strike_bias = fighter2['Sig_Strikes_Per Min'] / (fighter2['Takedown_Avg_Per Min'] + 0.1)
    
    # Style clash is already symmetric (uses absolute difference)
    features['style_clash'] = abs(fighter1_strike_bias - fighter2_strike_bias)
    
    # Ground game comparison
    f1_ground_offense = fighter1['Sub_Avg_Per_Min']
    f2_ground_defense = fighter2['Takedown_Def']
    f1_ground_advantage = f1_ground_offense - f2_ground_defense
    
    f2_ground_offense = fighter2['Sub_Avg_Per_Min']
    f1_ground_defense = fighter1['Takedown_Def']
    f2_ground_advantage = f2_ground_offense - f1_ground_defense
    
    # Symmetric ground advantage
    features['ground_advantage_abs_diff'] = abs(f1_ground_advantage - f2_ground_advantage)

    
    # Head striking comparison (symmetric)
    f1_head_pct = fighter1['Sig_Strikes_Head_Percent']
    f2_head_pct = fighter2['Sig_Strikes_Head_Percent']
    
    features['head_striking_abs_diff'] = abs(f1_head_pct - f2_head_pct)
    
   
    # Distance control comparison (symmetric)
    f1_distance_pct = fighter1['Sig_Strikes_While_Standing_Percent']
    f2_distance_pct = fighter2['Sig_Strikes_While_Standing_Percent']
    
    features['distance_control_abs_diff'] = abs(f1_distance_pct - f2_distance_pct)
    

    
    return features

In [23]:
def create_weight_class_relative_features(fighter, weight_class_stats_df):
    """Create features comparing fighter stats to weight class averages."""
    
    weight_class = fighter['Weight_Class']
    
    # Get weight class averages
    wc_avg = weight_class_stats_df[weight_class_stats_df['weight_class'] == weight_class].iloc[0]
    
    # Calculate relative features (how fighter compares to weight class average)
    rel_features = {}
    numerical_stats = ['Striking_Accuracy', 'Takedown_Accuracy', 'Sig_Strikes_Per Min', 
                      'Takedown_Avg_Per Min', 'Sig_Str_Def', 'Knockdown_Avg']
    
    for stat in numerical_stats:
        if stat in fighter and stat in wc_avg:
            # Convert to percentage above/below average
            rel_features[f'{stat}_rel_to_class'] = (fighter[stat] / wc_avg[stat]) - 1 
    
    return rel_features

# Create weight class statistics dataframe (this should be done once)
def calculate_weight_class_averages(fighters_df):
    """Calculate average statistics by weight class for normalization."""
    # Group by weight class and calculate means
    return fighters_df.groupby('Weight_Class').mean()

In [24]:
def create_location_features(fighter, fight_location):
    """Create features related to home advantage and travel."""
    
    features = {}
    
    # Extract country from location
    fight_country = fight_location.split(',')[-1].strip()
    
    # Check if birth place info exists
    if 'Place_of_Birth' in fighter and fighter['Place_of_Birth']:
        # Extract country of birth (basic approach)
        birth_country = fighter['Place_of_Birth'].split(',')[-1].strip()
        
        # Home advantage binary feature
        features['fighting_in_home_country'] = 1 if birth_country == fight_country else 0
        
        # Simple continent matching (would need mapping of countries to continents)
        fighter_continent = map_country_to_continent(birth_country)
        fight_continent = map_country_to_continent(fight_country)
        features['fighting_in_home_continent'] = 1 if fighter_continent == fight_continent else 0
    else:
        features['fighting_in_home_country'] = 0
        features['fighting_in_home_continent'] = 0
    
    return features

def map_country_to_continent(country):
    """Map country to continent - simplified example."""
    # This is a simplified mapping; ideally use a complete geography library
    continent_map = {
        'USA': 'North America',
        'Brazil': 'South America',
        'Russia': 'Europe',
    }
    return continent_map.get(country, 'Unknown')


In [25]:
def create_streak_features(fighter_name, results_df):
    """Create features related to winning/losing streaks and performance trends."""
    
    fighter_history = results_df[(results_df['FIGHTER_1'] == fighter_name) | 
                                 (results_df['FIGHTER_2'] == fighter_name)]
    
    if fighter_history.empty:
        return {
            'current_streak': 0,
            'finish_rate': 0,
            'ko_rate': 0,
            'sub_rate': 0,
            'decision_rate': 0,
            'round_efficiency': 0,
        }
    
    # Sort by date (most recent first)
    fighter_history = fighter_history.sort_values(by='DATE', ascending=False)
    
    # Calculate current streak (positive for win streak, negative for losing streak)
    streak = 0
    for idx, fight in fighter_history.iterrows():
        is_fighter1 = fight['FIGHTER_1'] == fighter_name
        result = fight['fighter_1_result'] if is_fighter1 else fight['fighter_2_result']
        
        if (result == 1 and streak >= 0) or (result == 0 and streak <= 0):
            if result == 1:
                streak += 1
            else:
                streak -= 1
        else:
            break
    
    # Calculate finish statistics
    total_fights = len(fighter_history)
    
    # Method breakdown
    ko_wins = sum(1 for _, f in fighter_history.iterrows() 
               if ((f['FIGHTER_1'] == fighter_name and f['fighter_1_result'] == 1) or 
                  (f['FIGHTER_2'] == fighter_name and f['fighter_2_result'] == 1)) 
               and f['method_label'] == 0)  # Assuming 0 is KO/TKO
    
    sub_wins = sum(1 for _, f in fighter_history.iterrows() 
                if ((f['FIGHTER_1'] == fighter_name and f['fighter_1_result'] == 1) or 
                   (f['FIGHTER_2'] == fighter_name and f['fighter_2_result'] == 1)) 
                and f['method_label'] == 1)  # Assuming 1 is submission
    
    dec_wins = sum(1 for _, f in fighter_history.iterrows() 
                if ((f['FIGHTER_1'] == fighter_name and f['fighter_1_result'] == 1) or 
                   (f['FIGHTER_2'] == fighter_name and f['fighter_2_result'] == 1)) 
                and f['method_label'] == 2)  # Assuming 2 is decision
    
    wins = ko_wins + sub_wins + dec_wins
    
    # Calculate round efficiency (average round of finish for wins)
    finish_rounds = [f['ROUND'] for _, f in fighter_history.iterrows() 
                    if ((f['FIGHTER_1'] == fighter_name and f['fighter_1_result'] == 1) or 
                       (f['FIGHTER_2'] == fighter_name and f['fighter_2_result'] == 1)) 
                    and f['method_label'] != 2]  # Not a decision
    
    round_efficiency = sum(finish_rounds) / len(finish_rounds) if finish_rounds else 0
    
    return {
        'current_streak': streak,
        'finish_rate': (ko_wins + sub_wins) / wins if wins > 0 else 0,
        'ko_rate': ko_wins / wins if wins > 0 else 0,
        'sub_rate': sub_wins / wins if wins > 0 else 0,
        'decision_rate': dec_wins / wins if wins > 0 else 0,
        'round_efficiency': round_efficiency,
    }

In [26]:
def calculate_reach_advantage(fighter1, fighter2, fighters_df):
    """Calculate reach advantage and normalize by weight class."""
    
    # Get fighter data
    f1_data = fighters_df[fighters_df['Name'] == fighter1]
    f2_data = fighters_df[fighters_df['Name'] == fighter2]
    
    if 'Reach' not in f1_data.columns or 'Reach' not in f2_data.columns:
        return 0  # Return 0 if reach data not available
    
    # Calculate reach difference
    reach_diff = f1_data['Reach'].values[0] - f2_data['Reach'].values[0]
    
    # Normalize by weight class (optional)
    weight_class = f1_data['Weight_Class'].values[0]
    weight_class_avg_reach = fighters_df[fighters_df['Weight_Class'] == weight_class]['Reach'].mean()
    
    # Normalize by dividing by the standard deviation of reach in that weight class
    weight_class_std_reach = fighters_df[fighters_df['Weight_Class'] == weight_class]['Reach'].std()
    
    # Return normalized reach advantage
    return reach_diff / weight_class_std_reach if weight_class_std_reach > 0 else reach_diff


In [27]:
def time_based_cv_split(results_df, n_splits=5):
    """Create time-based cross-validation splits to prevent data leakage."""
    
    # Sort by date
    results_df = results_df.sort_values(by='DATE')
    
    # Convert DATE to datetime if it's not already
    results_df['DATE'] = pd.to_datetime(results_df['DATE'])
    
    # Get unique dates
    unique_dates = results_df['DATE'].unique()
    
    # Create date splits
    date_splits = np.array_split(unique_dates, n_splits)
    
    # Create train/test indices for each split
    splits = []
    for i in range(1, len(date_splits)):
        # All fights before the current split are training
        train_dates = np.concatenate(date_splits[:i])
        # Current split dates are testing
        test_dates = date_splits[i]
        
        train_idx = results_df[results_df['DATE'].isin(train_dates)].index
        test_idx = results_df[results_df['DATE'].isin(test_dates)].index
        
        splits.append((train_idx, test_idx))
    
    return splits


In [29]:
def create_weight_class_specific_models(X, y, results_df, fighters_df):
    """Create separate models for each weight class for better predictions."""
    
    # Get all weight classes
    weight_classes = fighters_df['Weight_Class'].unique()
    
    models = {}
    scalers = {}
    
    for weight_class in weight_classes:
        print(f"Training model for {weight_class}...")
        
        # Get fights for this weight class
        weight_class_fights = results_df[results_df['weight_class'] == weight_class]
        
        if len(weight_class_fights) < 50:
            print(f"Not enough data for {weight_class}, using general model")
            continue
            
        # Get indices for these fights
        fight_indices = weight_class_fights.index
        
        # Get corresponding X, y data
        X_wc = X[fight_indices]
        y_wc = y[fight_indices]
        
        if len(np.unique(y_wc)) < 2:
            print(f"Not enough class diversity for {weight_class}, using general model")
            continue
            
        # Create and train model
        scaler = StandardScaler()
        X_wc_scaled = scaler.fit_transform(X_wc)
        
        model = LogisticRegression(max_iter=2000, solver='liblinear', random_state=42)
        model.fit(X_wc_scaled, y_wc)
        
        # Save model and scaler
        models[weight_class] = model
        scalers[weight_class] = scaler
        
    return models, scalers

In [ ]:
create_weight_class_specific_models()

In [31]:
def train_multiple_models(X, y):
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )
    X_train = StandardScaler().fit_transform(X_train)
    X_test = StandardScaler().fit_transform(X_test)
    print(f"Training models with {X_train.shape[0]} samples, {X_train.shape[1]} features")
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=2000,solver='liblinear', random_state=42),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=11),
        'SVM': SVC(probability=True,kernel='rbf',C=1, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
    }
    
    best_model = None
    best_accuracy = 0

    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        test_accuracy = accuracy_score(y_test, y_pred)
        
        cv_scores = cross_val_score(model, X, y, cv=5)
        cv_mean = cv_scores.mean()
        cv_std = cv_scores.std()
        
        print(f"{name} Test accuracy: {test_accuracy:.3f}")
        print(f"{name} CV performance: {cv_mean:.3f} ± {cv_std:.3f}")
        
        results[name] = {
            'model': model,
            'test_accuracy': test_accuracy,
            'cv_mean': cv_mean,
            'cv_std': cv_std,
            # 'classification_report': classification_report(y_test, y_pred)
        }
        
        if cv_mean > best_accuracy:
            best_accuracy = cv_mean
            best_model = model
    
    print("\n===== Model Comparison =====")
    for name, result in results.items():
        print(f"{name}: Test Acc={result['test_accuracy']:.3f}, CV={result['cv_mean']:.3f}±{result['cv_std']:.3f}")
    
    print(f"\nBest model: {max(results.items(), key=lambda x: x[1]['cv_mean'])[0]}")
    
    return results, best_model

In [ ]:
def run_complete_enhanced_pipeline(fighters_df, results_df, stats_df):
    """Run the complete pipeline from data preparation to model training."""
    
    print("Preparing training data with enhanced features...")
    X, y, feature_names, imputer = prepare_enhanced_training_data(results_df, fighters_df, stats_df)
    
    print(f"Generated {X.shape[1]} features for {X.shape[0]} fights")
    
    print("\nCreating time-based cross-validation splits...")
    cv_splits = create_time_based_cv_splits(results_df)
    
    print("\nTraining models with time-based cross-validation...")
    best_model, scaler, all_results = train_enhanced_model(X, y, cv_splits)
    
    # Save models and preprocessors
    print("\nSaving models and preprocessors...")
    joblib.dump(best_model, "best_model_enhanced.pkl")
    joblib.dump(scaler, "scaler_enhanced.pkl")
    joblib.dump(imputer, "imputer_enhanced.pkl")
    joblib.dump(feature_names, "feature_names_enhanced.pkl")
    
    print("\nPipeline complete! Models and preprocessors saved.")
    
    return best_model, scaler, imputer, feature_namesd


In [253]:
joblib.dump(best_model, f"best_model.pkl") 

['best_model.pkl']

In [254]:
best_model

GradientBoostingClassifier(random_state=42)

In [258]:
def test_prediction_model(fighters_df, results_df, loaded_model):
    """
    Test the prediction model with various fighter matchups and analyze results
    """
    print("\n===== UFC FIGHT PREDICTION MODEL TESTING =====\n")
    
    # 1. Test with known rivalries (historical matchups)
    print("\n----- TESTING KNOWN RIVALRIES -----")
    known_matchups = [
        ('conor mcgregor', 'nate diaz'),         # These two fought twice, split 1-1
        ('khabib nurmagomedov', 'conor mcgregor'), # Khabib won
        ('jon jones', 'daniel cormier'),          # Jones won twice
        ('israel adesanya', 'robert whittaker'),   # Adesanya won
        ('kamaru usman', 'colby covington'),      # Usman won twice
        ('alexander volkanovski', 'max holloway') # Volkanovski won multiple times
    ]
    
    print("\nHistorical Matchups:")
    for f1, f2 in known_matchups:
        try:
            winner, confidence = predict_fight(f1, f2, fighters_df, results_df, loaded_model)
            print(f"{f1} vs {f2}: Winner: {winner}, Confidence: {confidence:.2f}")
        except Exception as e:
            print(f"{f1} vs {f2}: Error - {str(e)}")
    
    # 2. Test with reversed order of same fighters
    print("\n----- TESTING ORDER SENSITIVITY -----")
    print("\nSame matchups with fighter order reversed:")
    for f1, f2 in known_matchups:
        try:
            winner1, conf1 = predict_fight(f1, f2, fighters_df, results_df, loaded_model)
            winner2, conf2 = predict_fight(f2, f1, fighters_df, results_df, loaded_model)
            
            if winner1 == f1 and winner2 == f2:
                print(f"{f1} vs {f2}: Model predicts whoever is first! This indicates a bias.")
            elif winner1 == winner2:
                print(f"{f1} vs {f2}: Model consistently predicts {winner1} regardless of order. Good!")
            else:
                print(f"{f1} vs {f2}: Inconsistent results when order is reversed!")
                
        except Exception as e:
            print(f"{f1} vs {f2}: Error - {str(e)}")
    
    # 3. Test with extreme mismatches (different weight classes)
    print("\n----- TESTING EXTREME MISMATCHES -----")
    mismatches = [
        ('jon jones', 'demetrious johnson'),      # Heavyweight vs Flyweight
        ('francis ngannou', 'petr yan'),          # Heavyweight vs Bantamweight
        ('kamaru usman', 'alexander volkanovski'), # Welterweight vs Featherweight
        ('weili zhang', 'cyril gane')            # Women's Strawweight vs Heavyweight
    ]
    
    print("\nExtreme Weight Mismatches:")
    for f1, f2 in mismatches:
        try:
            winner, confidence = predict_fight(f1, f2, fighters_df, results_df, loaded_model)
            print(f"{f1} vs {f2}: Winner: {winner}, Confidence: {confidence:.2f}")
            
            # Check if bigger fighter usually wins extreme mismatches
            f1_info = fighters_df[fighters_df['Name'] == f1].iloc[0]
            f2_info = fighters_df[fighters_df['Name'] == f2].iloc[0]
            if 'Weight_Class' in f1_info and 'Weight_Class' in f2_info:
                print(f"  {f1} Weight: {f1_info['Weight_Class']}, {f2} Weight: {f2_info['Weight_Class']}")
        except Exception as e:
            print(f"{f1} vs {f2}: Error - {str(e)}")
    
    # 4. Test with very similar fighters
    print("\n----- TESTING SIMILAR FIGHTERS -----")
    similar_fighters = [
        ('dustin poirier', 'justin gaethje'),     # Similar style brawlers
        ('israel adesanya', 'anderson silva'),     # Similar style strikers
        ('khabib nurmagomedov', 'islam makhachev'), # Similar style grapplers
        ('amanda nunes', 'valentina shevchenko')  # Similar level champs
    ]
    
    print("\nSimilar Fighters:")
    for f1, f2 in similar_fighters:
        try:
            winner, confidence = predict_fight(f1, f2, fighters_df, results_df, loaded_model)
            print(f"{f1} vs {f2}: Winner: {winner}, Confidence: {confidence:.2f}")
            
            # Check if confidence is lower for similar fighters (should be)
            if confidence < 0.6:
                print("  Low confidence as expected for similar fighters.")
            else:
                print("  Unusually high confidence for similar fighters.")
        except Exception as e:
            print(f"{f1} vs {f2}: Error - {str(e)}")
    
    # 5. Check for bias in predictions
    print("\n----- CHECKING FOR PREDICTION BIAS -----")
    
    # Generate random matchups
    import random
    random_pairs = []
    fighters = fighters_df['Name'].sample(40).tolist()
    
    for _ in range(20):
        f1 = random.choice(fighters)
        f2 = random.choice(fighters)
        if f1 != f2 and (f1, f2) not in random_pairs and (f2, f1) not in random_pairs:
            random_pairs.append((f1, f2))
    
    # Track statistics
    fighter1_wins = 0
    fighter2_wins = 0
    high_confidence_count = 0
    low_confidence_count = 0
    total_confidence = 0
    
    print("\nRandom Matchups:")
    for f1, f2 in random_pairs:
        try:
            winner, confidence = predict_fight(f1, f2, fighters_df, results_df, loaded_model)
            print(f"{f1} vs {f2}: Winner: {winner}, Confidence: {confidence:.2f}")
            
            if winner == f1:
                fighter1_wins += 1
            else:
                fighter2_wins += 1
                
            if confidence > 0.7:
                high_confidence_count += 1
            if confidence < 0.55:
                low_confidence_count += 1
                
            total_confidence += confidence
            
        except Exception as e:
            print(f"{f1} vs {f2}: Error - {str(e)}")
    
    # Print bias statistics
    print("\nBias Analysis:")
    print(f"Fighter 1 win rate: {fighter1_wins / len(random_pairs):.2f}")
    print(f"Fighter 2 win rate: {fighter2_wins / len(random_pairs):.2f}")
    print(f"Average confidence: {total_confidence / len(random_pairs):.2f}")
    print(f"High confidence predictions (>70%): {high_confidence_count} ({high_confidence_count/len(random_pairs):.2f})")
    print(f"Low confidence predictions (<55%): {low_confidence_count} ({low_confidence_count/len(random_pairs):.2f})")
    
    if fighter1_wins / len(random_pairs) > 0.7:
        print("WARNING: Model shows strong bias toward predicting Fighter 1 as the winner!")
        print("This suggests a problem with feature engineering or model training.")
    
    print("\n===== TEST COMPLETE =====")

# Run the tests
test_prediction_model(fighters_df, results_df, best_model)


===== UFC FIGHT PREDICTION MODEL TESTING =====


----- TESTING KNOWN RIVALRIES -----

Historical Matchups:
Feature count: 32
Raw prediction: [1], probs: [[1.91700334e-01 8.07908835e-01 3.90831176e-04]]
Scaled prediction: [1], probs: [[0.12860469 0.86802663 0.00336868]]
conor mcgregor vs nate diaz: Winner: conor mcgregor, Confidence: 0.81
Feature count: 32
Raw prediction: [1], probs: [[1.44623176e-01 8.55110340e-01 2.66483943e-04]]
Scaled prediction: [1], probs: [[0.12860469 0.86802663 0.00336868]]
khabib nurmagomedov vs conor mcgregor: Winner: khabib nurmagomedov, Confidence: 0.86
Feature count: 32
Raw prediction: [1], probs: [[0.08279686 0.91620299 0.00100015]]
Scaled prediction: [1], probs: [[0.12860469 0.86802663 0.00336868]]
jon jones vs daniel cormier: Winner: jon jones, Confidence: 0.92
Feature count: 32
Raw prediction: [1], probs: [[1.87527425e-01 8.12070695e-01 4.01880168e-04]]
Scaled prediction: [1], probs: [[0.12860469 0.86802663 0.00336868]]
israel adesanya vs robert whitta

In [64]:
for _, fight in skip_fights.iterrows():
    fighter_1 = fight['fighter1']
    fighter_2 = fight['fighter2']
    
    for i in range(1,3):
        fighter = fighter_1 if i == 1 else fighter_2
        if fighters_df[fighters_df['Name'] == fighter].empty:
            print(f"Fighter {fighter} not found in fighter data")


Fighter joel alvarez not found in fighter data
Fighter kiru sahota not found in fighter data
Fighter shi ming not found in fighter data
Fighter feng xiaocan not found in fighter data
Fighter lone'er kavanagh not found in fighter data
Fighter viviane araujo not found in fighter data
Fighter dusko todorovic not found in fighter data
Fighter zachary scroggin not found in fighter data
Fighter klaudia sygula not found in fighter data
Fighter ivana petrovic not found in fighter data
Fighter julianna pena not found in fighter data
Fighter jose aldo not found in fighter data
Fighter benoit saint denis not found in fighter data
Fighter fares ziam not found in fighter data
Fighter jessica andrade not found in fighter data
Fighter joel alvarez not found in fighter data
Fighter kaue fernandes not found in fighter data
Fighter caolan loughran not found in fighter data
Fighter jiri prochazka not found in fighter data
Fighter zach reese not found in fighter data
Fighter thiago moises not found in fig